In [1]:
%pip install -q torch tiktoken requests tqdm matplotlib llms-from-scratch

Note: you may need to restart the kernel to use updated packages.


In [2]:
import os
import json
import time
import torch
import tiktoken
import requests
from functools import partial
from torch.utils.data import Dataset, DataLoader

if torch.cuda.is_available():
    device = torch.device("cuda")
elif hasattr(torch.backends, "mps") and torch.backends.mps.is_available():
    device = torch.device("mps")
else:
    device = torch.device("cpu")

print("PyTorch:", torch.__version__)
print("Device:", device)

PyTorch: 2.5.1+cpu
Device: cpu


In [3]:
try:
    from llms_from_scratch.ch04 import GPTModel
    from llms_from_scratch.ch05 import (
        download_and_load_gpt2,
        load_weights_into_gpt,
        generate,
        text_to_token_ids,
        token_ids_to_text,
        calc_loss_loader,
        train_model_simple,
        plot_losses,
    )
    print("Imported helpers from llms_from_scratch.")
except ImportError:
    try:
        from previous_chapters import (
            GPTModel, load_weights_into_gpt, generate,
            text_to_token_ids, token_ids_to_text,
            calc_loss_loader, train_model_simple, plot_losses
        )
        from gpt_download import download_and_load_gpt2
        print("Imported helpers from local workshop files.")
    except ImportError as e:
        raise ImportError(
            "Could not import the workshop helpers. Make sure the package-install cell "
            "finished successfully, then restart the kernel and run all cells again. "
            "Alternatively, put previous_chapters.py and gpt_download.py beside this notebook."
        ) from e

Imported helpers from llms_from_scratch.


In [4]:
def download_json(file_path, url):
    if not os.path.exists(file_path):
        response = requests.get(url, timeout=60)
        response.raise_for_status()
        with open(file_path, "w", encoding="utf-8") as f:
            f.write(response.text)
    with open(file_path, "r", encoding="utf-8") as f:
        return json.load(f)

data_url = (
    "https://raw.githubusercontent.com/rasbt/LLMs-from-scratch/"
    "main/ch07/01_main-chapter-code/instruction-data.json"
)
data = download_json("instruction-data.json", data_url)
print("Number of instruction examples:", len(data))
print("Example keys:", list(data[0].keys()))

Number of instruction examples: 1100
Example keys: ['instruction', 'input', 'output']


In [5]:
def format_input(entry):
    prompt = (
        "Below is an instruction that describes a task. "
        "Write a response that appropriately completes the request."
        f"\n\n### Instruction:\n{entry['instruction']}"
    )
    if entry.get("input"):
        prompt += f"\n\n### Input:\n{entry['input']}"
    return prompt

print(format_input(data[0]))
print("\n### Response:\n", data[0]["output"])

Below is an instruction that describes a task. Write a response that appropriately completes the request.

### Instruction:
Evaluate the following phrase by transforming it into the spelling given.

### Input:
freind --> friend

### Response:
 The spelling of the given phrase "freind" is incorrect, the correct spelling is "friend".


In [6]:
# Keep the same 85% / 5% / 10% split used in the workshop.
train_end = int(len(data) * 0.85)
test_count = int(len(data) * 0.10)
train_data = data[:train_end]
test_data = data[train_end:train_end + test_count]
val_data = data[train_end + test_count:]

# For a first successful run on a laptop, use a small subset.
# Set MAX_TRAIN_EXAMPLES = None to use the complete training split.
MAX_TRAIN_EXAMPLES = 500
if MAX_TRAIN_EXAMPLES is not None:
    train_data = train_data[:MAX_TRAIN_EXAMPLES]
    val_data = val_data[:min(100, len(val_data))]
    test_data = test_data[:min(100, len(test_data))]

tokenizer = tiktoken.get_encoding("gpt2")
print("Train:", len(train_data), "| Validation:", len(val_data), "| Test:", len(test_data))
print("End-of-text token ID:", tokenizer.encode("<|endoftext|>", allowed_special={"<|endoftext|>"}))

Train: 500 | Validation: 55 | Test: 100
End-of-text token ID: [50256]


In [7]:
class InstructionDataset(Dataset):
    def __init__(self, rows, tokenizer):
        self.encoded_texts = []
        for entry in rows:
            full_text = format_input(entry) + f"\n\n### Response:\n{entry['output']}"
            self.encoded_texts.append(tokenizer.encode(full_text))

    def __len__(self):
        return len(self.encoded_texts)

    def __getitem__(self, index):
        return self.encoded_texts[index]


def custom_collate_fn(batch, pad_token_id=50256, ignore_index=-100,
                      allowed_max_length=1024, device="cpu"):
    max_len = min(max(len(item) + 1 for item in batch), allowed_max_length + 1)
    inputs_list, targets_list = [], []

    for item in batch:
        item = item[:max_len - 1]
        padded = item + [pad_token_id] * (max_len - len(item))
        inputs = torch.tensor(padded[:-1], dtype=torch.long)
        targets = torch.tensor(padded[1:], dtype=torch.long)

        # Ignore padding positions when calculating loss.
        targets[targets == pad_token_id] = ignore_index
        inputs_list.append(inputs)
        targets_list.append(targets)

    return torch.stack(inputs_list).to(device), torch.stack(targets_list).to(device)


batch_size = 2 if device.type == "cpu" else 4
collate_fn = partial(custom_collate_fn, device=device, allowed_max_length=512)
train_loader = DataLoader(InstructionDataset(train_data, tokenizer), batch_size=batch_size,
                          shuffle=True, drop_last=True, collate_fn=collate_fn)
val_loader = DataLoader(InstructionDataset(val_data, tokenizer), batch_size=batch_size,
                        shuffle=False, drop_last=False, collate_fn=collate_fn)
test_loader = DataLoader(InstructionDataset(test_data, tokenizer), batch_size=batch_size,
                         shuffle=False, drop_last=False, collate_fn=collate_fn)

first_inputs, first_targets = next(iter(train_loader))
print("Input batch:", tuple(first_inputs.shape))
print("Target batch:", tuple(first_targets.shape))

Input batch: (2, 60)
Target batch: (2, 60)


In [10]:
BASE_CONFIG = {
    "vocab_size": 50257,
    "context_length": 1024,
    "drop_rate": 0.0,
    "qkv_bias": True,
    "emb_dim": 768,
    "n_layers": 12,
    "n_heads": 12,
}
MODEL_SIZE = "124M"

settings, params = download_and_load_gpt2(model_size=MODEL_SIZE, models_dir="gpt2")
model = GPTModel(BASE_CONFIG)
load_weights_into_gpt(model, params)
model.to(device)
model.eval()

print("GPT-2 small loaded successfully.")
print("Parameter count (millions):", sum(p.numel() for p in model.parameters()) / 1e6)

File already exists and is up-to-date: gpt2\124M\checkpoint
File already exists and is up-to-date: gpt2\124M\encoder.json
File already exists and is up-to-date: gpt2\124M\hparams.json


model.ckpt.data-00000-of-00001: 100%|██████████| 498M/498M [12:25<00:00, 668kiB/s]     


File already exists and is up-to-date: gpt2\124M\model.ckpt.index
File already exists and is up-to-date: gpt2\124M\model.ckpt.meta
File already exists and is up-to-date: gpt2\124M\vocab.bpe
GPT-2 small loaded successfully.
Parameter count (millions): 163.037184


In [11]:
prompt = format_input(val_data[0])
model.eval()
with torch.no_grad():
    token_ids = generate(
        model=model,
        idx=text_to_token_ids(prompt, tokenizer).to(device),
        max_new_tokens=40,
        context_size=BASE_CONFIG["context_length"],
        eos_id=50256,
    )
generated = token_ids_to_text(token_ids, tokenizer)
print(generated[len(prompt):].strip())

### Instruction:

Convert the active sentence to passive: 'The chef cooks the meal every day.'

### Instruction:

Convert the active sentence to passive: '


In [12]:
model.eval()
with torch.no_grad():
    train_loss = calc_loss_loader(train_loader, model, device, num_batches=min(2, len(train_loader)))
    val_loss = calc_loss_loader(val_loader, model, device, num_batches=min(2, len(val_loader)))
print(f"Initial train loss: {train_loss:.3f}")
print(f"Initial validation loss: {val_loss:.3f}")

Initial train loss: 4.007
Initial validation loss: 4.219


In [13]:
torch.manual_seed(123)
optimizer = torch.optim.AdamW(model.parameters(), lr=5e-5, weight_decay=0.1)
num_epochs = 1
start_time = time.time()

train_losses, val_losses, tokens_seen = train_model_simple(
    model, train_loader, val_loader, optimizer, device,
    num_epochs=num_epochs,
    eval_freq=max(1, len(train_loader) // 2),
    eval_iter=min(2, len(val_loader)),
    start_context=format_input(val_data[0]),
    tokenizer=tokenizer,
)

print(f"Fine-tuning finished in {(time.time() - start_time) / 60:.2f} minutes.")

Ep 1 (Step 000000): Train loss 2.992, Val loss 3.357
Ep 1 (Step 000125): Train loss 0.598, Val loss 1.013
Below is an instruction that describes a task. Write a response that appropriately completes the request.  ### Instruction: Convert the active sentence to passive: 'The chef cooks the meal every day.'  ### Response: The chef cooks the meal every day.  ### Input: The chef cooks the meal every day.  ### Response: The chef cooks the meal every day.  ### Input: The chef
Fine-tuning finished in 4.98 minutes.


In [14]:
model.eval()
for entry in test_data[:3]:
    prompt = format_input(entry)
    with torch.no_grad():
        token_ids = generate(
            model=model,
            idx=text_to_token_ids(prompt, tokenizer).to(device),
            max_new_tokens=100,
            context_size=BASE_CONFIG["context_length"],
            eos_id=50256,
        )
    output = token_ids_to_text(token_ids, tokenizer)
    response = output[len(prompt):].replace("### Response:", "").strip()
    print("INSTRUCTION:", entry["instruction"])
    print("MODEL RESPONSE:", response)
    print("-" * 60)

INSTRUCTION: Rewrite the sentence using a simile.
MODEL RESPONSE: The car is very fast.

### Input:
The car is very fast.


The car is very fast.

### Input:
The car is very fast.


The car is very fast.

### Input:
The car is very fast.


The car is very fast.

### Input:
The car is very fast.

### Response
------------------------------------------------------------
INSTRUCTION: What type of cloud is typically associated with thunderstorms?
MODEL RESPONSE: A type of cloud is typically associated with thunderstorms. It is a type of precipitation. It is a type of precipitation that is typically associated with thunderstorms. It is a type of precipitation that is typically associated with thunderstorms.


A type of cloud is typically associated with thunderstorms. It is a type of precipitation. It is a type of precipitation that is typically associated with thunderstorms.


A type of cloud
------------------------------------------------------------
INSTRUCTION: Name the author of 'Prid

In [15]:
save_path = "gpt2-small-instruction-finetuned.pth"
torch.save(model.state_dict(), save_path)
print("Saved model weights to:", os.path.abspath(save_path))

Saved model weights to: C:\LLM-workshop\LLMs-from-scratch-main\ch07\01_main-chapter-code\gpt2-small-instruction-finetuned.pth
